In [ ]:
# 모델 적용하기

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import pickle
import torch.nn as nn
from PIL import ImageFont, ImageDraw, Image
from tensorflow.keras.models import load_model
import json
from utils.gemini import AIChatClient
from dotenv import load_dotenv
import os

In [ ]:
# ### label - idx mapping정보 가져오기
# import pickle
# with open('../data/label_to_idx.pickle', 'rb') as f:
#     label_to_idx = pickle.load(f)
# print(label_to_idx)
# idx_to_label = {value : key for key, value in label_to_idx.items()} ## idx로 label접근


FEATURE_DIM = 84
NUM_CLASSES = 133
MIN_SEQ = 40
MAX_SEQ = 101

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_API_URL = os.getenv("GEMINI_API_URL")
LLM_ID = "gemini-2.0-flash-lite"


In [ ]:
fontpath = "AppleGothic.ttf"
font = ImageFont.truetype(fontpath, 40)

with open('idx2word2.json', 'r', encoding='utf-8') as f:
    idx2word = json.load(f)

In [ ]:
def norm(x):
    mean = x.mean(axis=0, keepdims=True)
    std  = x.std(axis=0, keepdims=True) + 1e-6
    x = (x - mean) / std
    return x

In [ ]:
def extract_keypoints(results):
        # 왼손 키포인트 추출
        joint_left = np.zeros((21,2))
        if results.left_hand_landmarks:
            # 키포인트 추출
            for j, lm in enumerate(results.left_hand_landmarks.landmark):
                joint_left[j] = [lm.x, lm.y]
        hand_left_x = norm(joint_left[:, 0])
        hand_left_y = norm(joint_left[:, 1])          
        
        # 오른손 키포인트 추출
        joint_right = np.zeros((21,2))
        if results.right_hand_landmarks:
            # 키포인트 추출
            for j, lm in enumerate(results.right_hand_landmarks.landmark):
                joint_right[j] = [lm.x, lm.y]
        hand_right_x = norm(joint_right[:, 0])
        hand_right_y = norm(joint_right[:, 1])
        
        hand_left_xy = np.concatenate([hand_left_x, hand_left_y], axis=0)
        hand_right_xy = np.concatenate([hand_right_x, hand_right_y], axis=0)
        
        frame_keypoints =  np.concatenate([hand_left_xy.flatten(), hand_right_xy.flatten()], axis=0)

        return frame_keypoints # d =  21 * 4 = 84

In [ ]:
# video_path = '/Users/hyeonji/Downloads/수어 영상/1.Training/11(F)/NIA_SL_WORD0007_REAL11_F.mp4'  # 처리할 영상 파일 경로
# cap = cv2.VideoCapture(video_path)
# 비디오
cap = cv2.VideoCapture(0)

# 웹캠 프레임 크기를 정사각형으로 설정
frame_size = 640
cap.set(cv2.CAP_PROP_FRAME_WIDTH, frame_size)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, frame_size)


# holistic설정
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic()
mp_draw = mp.solutions.drawing_utils
mp_draw_styles = mp.solutions.drawing_styles

In [ ]:
def keypoints_padding(keypoints):        
    pad_len = MAX_SEQ - len(keypoints)
    pad_array = np.zeros((pad_len, FEATURE_DIM), dtype=np.float32)
    keypoints = np.vstack([keypoints, pad_array])
    return keypoints.astype(np.float32)

In [ ]:
### 모델 가져오기 ###
model = load_model('saved_model/gru_tensor3_mask_92/gru_tensor.keras')
### gemini 설정 ###
chat_client = AIChatClient(GEMINI_API_URL, GEMINI_API_KEY, LLM_ID)

keypoints_sequence = [] # 키포인트 저장용

seq_action = ["", ""] # 확인용
sentence = [" ", ] # 예측한 단어 저장용
word = ""

# 이미지 입력 캡처 및 처리
# media pipe 는 RGB
while cap.isOpened():
    success, image = cap.read()
    imageRGB = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = holistic.process(imageRGB)

    # print("왼손 랜드마크: ", results.left_hand_landmarks)
    # print("오른손 랜드마크: ", results.right_hand_landmarks)
    # print("얼굴 랜드마크: ", results.face_landmarks)
    # print("pose 랜드마크: ", results.pose_landmarks)

    angles = extract_keypoints(results)
    keypoints_sequence.append(angles)
    if len(keypoints_sequence) > MAX_SEQ: # 101 초과이면 마지막 100로 prediction 한다
        sequence = keypoints_sequence[-MAX_SEQ:]  
    else: # 101개 이하일 때 패딩해서 사용
        sequence = keypoints_sequence
        sequence = keypoints_padding(sequence)
    # print(f"현재 시퀀스 길이: {len(keypoints_sequence)}")

    if len(keypoints_sequence) >= MIN_SEQ:  # 전체 30 프레임 이상이면 
        sequence = np.expand_dims(sequence, axis=0)  # batch dimension 추가
        predictions = model.predict(sequence, verbose=0)
        predicted_class = np.argmax(predictions, axis=1)[0]
        
        predicted_class_name = idx2word.get(str(predicted_class), "<UNK>")
        probality = predictions[0][predicted_class]
        if probality > 0.5:
            print(f"예측된 클래스: {predicted_class_name}, 확률: {probality:.2f}")

        if probality>0.8:
            seq_action.append(predicted_class_name)
            if seq_action[-1] == seq_action[-2]: # 연속으로 같은 동작일 때 (정확성 체크)
                if predicted_class_name != sentence[-1]:
                    sentence.append(predicted_class_name)
                    print('sentence ', sentence)
                    keypoints_sequence.clear()

                    print('prediction ', predicted_class, ':', predicted_class_name)
                    print('acc ', probality) # (batch, class)

                    word = f'Prediction: {predicted_class_name} Acc: {probality:.2f}'    

    # 점 그리기
    annotated_image = image.copy()
    mp_draw.draw_landmarks(annotated_image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_draw.draw_landmarks(annotated_image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

    # 한글 폰트 출력 
    img_pil = Image.fromarray(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)) # cv -> PIL
    draw = ImageDraw.Draw(img_pil)
    draw.text((10, 30), f'{word.upper()}', font=font, fill=(0, 0, 0))
    img = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR) # PIL -> cv

    cv2.imshow('output', img)
    
    if cv2.waitKey(1) & 0xFF == 13:  # 엔터 키를 누르면 문장 만든다
        print("[Enter] sentence : ", sentence)
        if len(sentence) > 1:
            results = chat_client.ask(sentence)
            print("Gemini 응답: ", results)
            word = results
            results = chat_client.translate(results, "영어") # 다른언어도 가능
            print("번역된 응답: ", results)
            word = word + "\n" + "영어 번역:" + results
        # 예측 단어리스트 초기화
        sentence = ["",]
        # word = ""

    # ESC키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == 27:
        break



cap.release()
holistic.close()
cv2.destroyAllWindows()

